In [1]:
import os
import requests
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain.agents import AgentExecutor, create_tool_calling_agent
from langchain_core.tools import tool
from langchain.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from currentsapi import CurrentsAPI


load_dotenv()

os.environ["OPEN_AI_API"] = os.getenv("OPENAI_API_KEY")
os.environ["CURRENTS_API_KEY"] = os.getenv("CURRENTS_API_KEY")
os.environ["LANGCHAIN_API_KEY"] = os.getenv("LANGCHAIN_API_KEY")
os.environ["LANGCHAIN_TRACING_V2"] = "true"


In [2]:
llm = ChatOpenAI(model="gpt-4o")
output_parser = StrOutputParser()

# response = llm.invoke("Hey! Tell me something about you")
# print(response.content)

In [3]:
api_key = os.environ["CURRENTS_API_KEY"]
news = CurrentsAPI(api_key=api_key)


In [4]:
news.search(keywords='dhaka')

{'status': 'ok',
 'news': [{'id': '8efd3298-3b76-46b8-9155-c3c406bf98e4',
   'title': "Bangladesh To Probe Attack On Taslima Nasreen's Publisher At Book Fair",
   'description': 'Nobel laureate Muhammad Yunus-led interim government in Bangladesh has ordered a probe after a group of madrasa students attacked a stall at the Amar Ekushey Book Fair in Dhaka over the display of a book written by exiled author Taslima Nasreen.',
   'url': 'https://www.ndtv.com/world-news/bangladesh-to-probe-attack-on-taslima-nasreens-publisher-at-book-fair-7683453',
   'author': 'ndtv',
   'image': 'https://c.ndtvimg.com/2025-02/shu8brv8_taslima-nasreen_625x300_11_February_25.jpeg?im=FeatureCrop,algorithm=dnn,width=1200,height=738',
   'language': 'en',
   'category': ['world'],
   'published': '2025-02-11 05:32:10 +0000'},
  {'id': 'c3d0b10c-1bcf-433e-9d1c-3761d496d282',
   'title': "Bangladesh Launches 'Operation Devil Hunt' After Attack On Ex-Minister's House",
   'description': 'Bangladesh\'s interim gov

In [5]:
prompt = ChatPromptTemplate.from_template("""
You are a smart news assistant. Use the tools to fetch the latest news based on the provided keywords, city, or country. 
Along with presenting the news, add some creative commentary or insights to engage the user. 
Conclude your response with a brief summary or key takeaway.

If you receive only the city name, intelligently determine the country code without asking the user. 
If unsure, make an educated guess based on common knowledge.

Read all the news you got and analyze these to make a best output

Question: {question}

{agent_scratchpad}
""")


In [9]:
question = "What is the current news of dhaka"

In [8]:
@tool
def get_current_news(keyword: str) -> str:
    """
    Fetches the latest news articles based on the provided keyword.
    """
    currents = CurrentsAPI(api_key=os.environ["CURRENTS_API_KEY"])
    response = currents.search(keywords=keyword)
    
    if response['status'] == 'ok':
        news_list = []
        for article in response['news']:
            news_item = f"Title: {article['title']}\nDescription: {article['description']}\nURL: {article['url']}\nPublished: {article['published']}\n{'-' * 50}"
            news_list.append(news_item)
        return "\n".join(news_list)
    else:
        return "Failed to fetch news."

In [11]:
# get_current_news("dhaka")

In [12]:
# Binding tools
tools = [get_current_news]
llm_with_tools = llm.bind_tools(tools)

# Creating the ReAct agent
agent = create_tool_calling_agent(llm_with_tools, tools, prompt)

# Agent Executor to manage tool execution
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True)

# Now invoking the agent
response = agent_executor.invoke({"question": question})

print(response['output'])  # To get the final response content




> Entering new AgentExecutor chain...

Invoking: `get_current_news` with `{'keyword': 'dhaka'}`


Title: Bangladesh To Probe Attack On Taslima Nasreen's Publisher At Book Fair
Description: Nobel laureate Muhammad Yunus-led interim government in Bangladesh has ordered a probe after a group of madrasa students attacked a stall at the Amar Ekushey Book Fair in Dhaka over the display of a book written by exiled author Taslima Nasreen.
URL: https://www.ndtv.com/world-news/bangladesh-to-probe-attack-on-taslima-nasreens-publisher-at-book-fair-7683453
Published: 2025-02-11 05:32:10 +0000
--------------------------------------------------
Title: Bangladesh Launches 'Operation Devil Hunt' After Attack On Ex-Minister's House
Description: Bangladesh's interim government launched an operation Saturday after a student group gave a 24-hour ultimatum to track down "culprits" who assaulted their activists during a reported attack by protesters on an Awami League leader's house near Dhaka.
URL: https:

## Prompt for travel

In [6]:
prompt = ChatPromptTemplate.from_template("""
You are a smart news assistant. Use the tools to fetch the latest news based on the provided keywords.

Analize the news and give a 2-3 sentance summary. If the user ask for city provide a suggestion that is it safe for visit or not based on the news

Question: {question}

{agent_scratchpad}
""")
question = "Tell me about dhaka"

In [9]:
# Binding tools
tools = [get_current_news]
llm_with_tools = llm.bind_tools(tools)

# Creating the ReAct agent
agent = create_tool_calling_agent(llm_with_tools, tools, prompt)

# Agent Executor to manage tool execution
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True)

# Now invoking the agent
response = agent_executor.invoke({"question": question})

print(response['output'])  # To get the final response content




> Entering new AgentExecutor chain...

Invoking: `get_current_news` with `{'keyword': 'Dhaka'}`


Title: Bangladesh To Probe Attack On Taslima Nasreen's Publisher At Book Fair
Description: Nobel laureate Muhammad Yunus-led interim government in Bangladesh has ordered a probe after a group of madrasa students attacked a stall at the Amar Ekushey Book Fair in Dhaka over the display of a book written by exiled author Taslima Nasreen.
URL: https://www.ndtv.com/world-news/bangladesh-to-probe-attack-on-taslima-nasreens-publisher-at-book-fair-7683453
Published: 2025-02-11 05:32:10 +0000
--------------------------------------------------
Title: Bangladesh Launches 'Operation Devil Hunt' After Attack On Ex-Minister's House
Description: Bangladesh's interim government launched an operation Saturday after a student group gave a 24-hour ultimatum to track down "culprits" who assaulted their activists during a reported attack by protesters on an Awami League leader's house near Dhaka.
URL: https: